In [1]:
import guardrails
print("success")

sucess


In [3]:
from guardrails import Guard, OnFailAction
from guardrails.validators import Validator, PassResult,FailResult,register_validator

@register_validator(name="no-direct-approval", data_type="string")
class NoDirectApproval(Validator): 
    def _validate(self,value,metadata):
        if "without it approval" in value.lower():
                return FailResult(error_message="Request blocked: Software installation requires IT approval. Please contact IT for assistance")
        return PassResult()
guard=Guard().use(NoDirectApproval(on_fail= OnFailAction.NOOP))

result1 = guard.validate("How do I reset my password?")
print(result1.validation_passed) 


result2 = guard.validate(
    "How can I install software without IT approval?"
)
print(result2.validation_passed) 

True
False


adding fix value : let the validator make the text proper

In [ ]:
import string
@register_validator(name="change-the-prompt-value", data_type="string")
class ChangeQuestionText(Validator):
    def _validate(self,value,metadata):
        if "without it approval" in value.lower():
            replace_text=value.replace("without it approval","with it approval")
            return FailResult(error_message="Request blocked: Software installation requires IT approval.",fix_value=replace_text,)
        return PassResult()
guard=Guard().use(ChangeQuestionText(on_fail= OnFailAction.FIX))

result2 = guard.validate(
    "How can I install software without it approval?"
)
print("passed",result2.validation_passed) 
print("validated output",result2.validated_output)


passed True
validated output How can I install software with it approval?


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


In [ ]:
from guardrails import OnFailAction
from guardrails.errors import ValidationError
text="How can I install software without it approval?"
act=[
    OnFailAction.FIX,
    OnFailAction.FILTER,
    OnFailAction.REFRAIN,
    OnFailAction.EXCEPTION,
    OnFailAction.NOOP,   
]
for i in act:
    guard=Guard().use(ChangeQuestionText(on_fail=i))
    try:
        result=guard.validate(text)
        print(i,"Passed",result.validation_passed,"| Output :",result.validated_output)
    except ValidationError:
            print(i,"-->raised ValidationError,blocked")

OnFailAction.FIX Passed True | Output : How can I install software with it approval?
OnFailAction.FILTER Passed False | Output : None
OnFailAction.REFRAIN Passed False | Output : None
OnFailAction.EXCEPTION -->raised ValidationError,blocked
OnFailAction.NOOP Passed False | Output : How can I install software without it approval?


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.
